<div dir="rtl">
<center>
  <p>
    <font size="6">
      <b>« تمرین برنامه‌نویسی اول »</b>
    </font>
  </p>

  <p>
    <font size="4">
     یافتن دادگان مشابه با استفاده از الگوریتم LSH
    </font>
  </p>

  <p style="margin-top: 15px;">
    <font size="3" color="#102C52">
      <b>نام و نام خانوادگی:</b> علیرضا جهانبخش &nbsp;&nbsp;|&nbsp;&nbsp; <b>شماره دانشجویی:</b> ۴۰۴۴۲۲۲۰۶
    </font>
  </p>

</center>
</div>



<div dir="rtl">
در این تمرین، هدف یافتن متن‌های مشابه در مجموعه‌داده نظرات کاربران درباره داروها با استفاده از الگوریتم Locality-Sensitive Hashing (LSH) است. ابتدا متن‌ها با استفاده از یک مدل زبانی به بردارهای عددی (Embedding) تبدیل می‌شوند.

برخلاف روش MinHash-LSH که برای شباهت Jaccard استفاده می‌شود، در این تمرین از **Random-Hyperplane LSH** استفاده می‌کنیم که برای بردارها و شباهت کسینوسی مناسب است.

در این روش، تعدادی ابرصفحه تصادفی تولید می‌شود. هر ابرصفحه بر اساس علامت ضرب داخلی بردار ورودی با بردار عمود بر ابرصفحه، یک بیت صفر یا یک ایجاد می‌کند. ترکیب این بیت‌ها شناسه یک باکت (Bucket) را تشکیل می‌دهد.

بردارهایی که از نظر جهت به یکدیگر نزدیک‌ترند، احتمال بیشتری دارند که در یک باکت قرار بگیرند. با استفاده از چند جدول هش مستقل، کاندیدهای مشابه بازیابی شده و سپس با معیار شباهت کسینوسی مقایسه و رتبه‌بندی می‌شوند.

این روش تقریبی است؛ بنابراین ممکن است برخی از نزدیک‌ترین بردارها در مجموعه کاندیدها قرار نگیرند.
</div>



<div dir="rtl">
برای اجرای سریع‌تر نوت‌بوک، استفاده از GPU در محیط Google Colab توصیه می‌شود. اجرای برنامه روی CPU نیز امکان‌پذیر است، اما تولید embeddingها ممکن است زمان بیشتری نیاز داشته باشد.

بخش‌های `TODO` را تکمیل کنید.
</div>


<div dir=rtl>
ابتدا کتابخانه‌های مورد نیاز را نصب می‌کنیم.
</div>

In [1]:
%pip -q install "datasets>=3,<5" "sentence-transformers>=3,<6"

<div dir=rtl>
کتابخانه‌های مورد نیاز را import می‌کنیم.
</div>

In [2]:
import hashlib
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from numpy import dot
from numpy.linalg import norm
from sentence_transformers import SentenceTransformer

DATASET_ID = "lewtun/drug-reviews"
DATASET_SPLIT = "train"
MODEL_ID = "sentence-transformers/all-mpnet-base-v2"
NUM_DOCUMENTS = 25971
RANDOM_SEED = 42
DATA_PATH = Path("/content/medical_problems.csv") if Path("/content").is_dir() else Path.cwd() / "medical_problems.csv"
REFERENCE_SHA256 = "86195008170767ed2ec4de50c9b4e5ac74c11e9930ef004855d58255bea5be92"

print("Device:", "GPU" if torch.cuda.is_available() else "CPU (may be slower)")
print("Source dataset:", DATASET_ID, "| split:", DATASET_SPLIT)


<div dir=rtl>
با استفاده از دستور زیر، مجموعه داده را از Hugging Face دانلود کرده و فایل `medical_problems.csv` را ایجاد می‌کنیم.
</div>

In [3]:
if DATA_PATH.exists() and DATA_PATH.stat().st_size > 0:
    current_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
else:
    current_hash = None

if current_hash == REFERENCE_SHA256:
    print("Dataset already prepared; using validated local copy:", DATA_PATH)
else:
    print("Downloading the training split from Hugging Face...")
    try:
        raw_dataset = load_dataset(DATASET_ID, split=DATASET_SPLIT)
    except Exception as exc:
        raise RuntimeError(
            "دانلود دیتاست ناموفق بود. اتصال اینترنت Colab و دسترسی به huggingface.co "
            "را بررسی کنید و سپس این سلول را دوباره اجرا کنید. "
            "آدرس منبع: https://huggingface.co/datasets/lewtun/drug-reviews"
        ) from exc

    source_df = raw_dataset.to_pandas()
    required_columns = {"review", "condition"}
    missing_columns = required_columns - set(source_df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns in the source dataset: {sorted(missing_columns)}")

    prepared = source_df[["review", "condition"]].dropna().copy()
    for column in ("review", "condition"):
        prepared[column] = prepared[column].astype(str).str.strip()
    prepared = prepared.loc[prepared["review"].ne("") & prepared["condition"].ne("")]
    prepared = prepared.drop_duplicates(subset=["review"], keep="first")
    if len(prepared) < NUM_DOCUMENTS:
        raise ValueError(f"Only {len(prepared)} usable rows; at least {NUM_DOCUMENTS} are required.")

    prepared = prepared.sample(n=NUM_DOCUMENTS, random_state=RANDOM_SEED).reset_index(drop=True)
    prepared.to_csv(DATA_PATH, index=False, encoding="utf-8")
    current_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
    del source_df, prepared, raw_dataset

print("CSV created:", DATA_PATH)
print("SHA256:", current_hash)
if current_hash != REFERENCE_SHA256:
    print("WARNING: This CSV differs from the instructor's reference version.")
    print("The source dataset may have been updated. Inform the instructor before comparing exact results.")


CSV created: medical_problems.csv
SHA256: 86195008170767ed2ec4de50c9b4e5ac74c11e9930ef004855d58255bea5be92


<div dir=rtl>
مجموعه داده را با استفاده از کتابخانه pandas بارگذاری می‌کنیم.
</div>

In [4]:
data = pd.read_csv(DATA_PATH)
required_columns = {"review", "condition"}
missing_columns = required_columns - set(data.columns)
if missing_columns:
    raise ValueError(f"Required columns missing: {sorted(missing_columns)}")

if len(data) != NUM_DOCUMENTS:
    raise ValueError(f"Expected {NUM_DOCUMENTS} records, got {len(data)}")
if data[["review", "condition"]].isna().any().any():
    raise ValueError("Some reviews or conditions are missing")
if data["review"].duplicated().any():
    raise ValueError("Duplicate review text detected")

data = data.reset_index(drop=True)
print("Dataset shape:", data.shape)
print("Columns:", data.columns.tolist())
print("First examples:")
display(data.head(3))


Dataset shape: (25971, 2)
Columns: ['review', 'condition']
First examples:


,review,condition
0,"""Both of my feet need to be fused because of a...",Osteoarthritis
1,"""Tramadol is my miracle drug! 50mg 4-6 times a...",Restless Legs Syndrome
2,"""I have issues with recurring bacterial vagini...",Bacterial Infection


<div dir=rtl>
مدل ‌Embedding را بارگذاری می‌کنیم.
</div>

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

try:
    model = SentenceTransformer(MODEL_ID, device=device)
except Exception as exc:
    raise RuntimeError(
        f"مدل {MODEL_ID} دریافت یا بارگذاری نشد. "
        "اتصال اینترنت و دسترسی به Hugging Face را بررسی کنید؛ "
        "سپس این سلول را دوباره اجرا کنید."
    ) from exc

print("Model:", MODEL_ID)
print("Device:", device)
print("Embedding dimension:", model.get_sentence_embedding_dimension())


Model: sentence-transformers/all-mpnet-base-v2
Device: cuda
Embedding dimension: 768


<div dir=rtl>
در این مرحله برای متن‌های ستون `review`، بردار embedding ایجاد می‌کنیم.
</div>

In [6]:
batch_size = 64 if device == "cuda" else 16
start_encoding = time.perf_counter()
complaint_embeddings = model.encode(
    data["review"].tolist(),
    batch_size=batch_size,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype(np.float32, copy=False)
encoding_seconds = time.perf_counter() - start_encoding

assert complaint_embeddings.shape == (NUM_DOCUMENTS, 768), complaint_embeddings.shape
assert np.isfinite(complaint_embeddings).all(), "Invalid embedding values"
print(f"Embedding time: {encoding_seconds:.2f} s")
print("Embedding shape:", complaint_embeddings.shape)


Embedding time: 338.07 s
Embedding shape: (25971, 768)


<div dir=rtl>
ساختار embeddingها و میزان حافظه اشغال‌شده را مشاهده می‌کنیم.
</div>

In [7]:
complaint_embeddings.shape


(25971, 768)

In [8]:
round(complaint_embeddings.nbytes / 1024 / 1024, 2)


76.09

<div dir=rtl>
تعداد hyperplaneها و جداول hash را مشخص می‌کنیم.
</div>

In [9]:
search_space_len = complaint_embeddings.shape[0]
embedding_dims = complaint_embeddings.shape[1]

target_n_buckets = search_space_len / 16
target_n_buckets


1623.1875

In [10]:
n_planes = 11
num_hash_table = 25
print('Possible buckets per table:', 2 ** n_planes)


Possible buckets per table: 2048


<div dir=rtl>
برای هر جدول hash، مجموعه‌ای از hyperplaneهای تصادفی تولید می‌کنیم.
</div>

In [11]:
rng = np.random.default_rng(42)
planes_l = [rng.normal(size=(embedding_dims, n_planes)) for _ in range(num_hash_table)]
print('Number of tables:', len(planes_l))
print('Planes matrix shape:', planes_l[0].shape)


Number of tables: 25
Planes matrix shape: (768, 11)



<div dir="rtl">
سؤال اول) تابع `hash_vector` را با استفاده از روش Random-Hyperplane LSH تکمیل کنید، به‌طوری‌که برای هر ابرصفحه یک بیت تولید شده و ترکیب بیت‌ها به یک شناسه هش از نوع integer تبدیل شود.

فرض کنید ضرب داخلی صفر یا مثبت معادل بیت ۱ و ضرب داخلی منفی معادل بیت ۰ است. همچنین بیت مربوط به اولین ابرصفحه را کم‌ارزش‌ترین بیت در نظر بگیرید.
</div>


In [12]:
def hash_vector(v, planes):
    projections = np.dot(v, planes)
    hash_key = 0
    for i, val in enumerate(projections):
        if val >= 0:
            hash_key |= (1 << i)
    return int(hash_key)


<div dir=rtl>
سوال دوم) قسمت‌های `TODO` تابع زیر را تکمیل کنید.
</div>

In [13]:
def make_hash_tables(search_space, planes):
    num_of_planes = planes.shape[1]
    num_buckets = 2 ** num_of_planes
    hash_table = {i: [] for i in range(num_buckets)}
    id_table = {i: [] for i in range(num_buckets)}

    for i, v in enumerate(search_space):
        b_id = hash_vector(v, planes)
        hash_table[b_id].append(v)
        id_table[b_id].append(i)

    return hash_table, id_table


<div dir=rtl>
سوال سوم) با استفاده از تابع مرحله قبل، ۲۵ جدول hash ایجاد کنید.
</div>

In [14]:
hash_tables = []
id_tables = []

for planes in planes_l:
    h_tbl, i_tbl = make_hash_tables(complaint_embeddings, planes)
    hash_tables.append(h_tbl)
    id_tables.append(i_tbl)


<div dir=rtl>
سوال چهارم) نحوه کار کد زیر را به طور دقیق توضیح دهید.
</div>

<div dir="rtl" style="text-align: right; line-height: 1.8;">

### پاسخ سؤال چهارم: تحلیل گام‌به‌گام عملکرد تابع `reduce_search_space`

این تابع وظیفه تقلیل هوشمندانه فضای جست‌وجوی پربعد را بر عهده دارد:

1. **اعتبارسنجی ابعاد و ساختارها:** ابتدا بررسی می‌کند که تعداد مجموعه‌های ابرصفحه با تعداد جداول هش و جداول شناسه تطابق کامل داشته باشد.
2. **پیمایش موازی جداول هش:** به ازای هر یک از ۲۵ جدول هش مستقل، شناسه باکت بردار پرس‌وجو (`v`) را با استفاده از ابرصفحه‌های همان جدول از طریق تابع `hash_vector` محاسبه می‌کند.
3. **استخراج اسناد هم‌باکت:** تمام بردارهای امبدینگ و شناسه‌های اسنادی که در آن جدول درون همان باکت قرار گرفته‌اند را بازیابی می‌کند.
4. **حذف موارد تکراری (Deduplication):** با استفاده از یک مجموعه (`candidate_ids_set`)، از ورود شناسه‌های تکراری (اسنادی که در بیش از یک جدول با پرس‌وجو هم‌باکت شده‌اند) جلوگیری کرده و اجتماع اعضای کاندید را تشکیل می‌دهد.
5. **خروجی بهینه:** در نهایت، زیرفضای محدودشده اسناد کاندید را به عنوان یک آرایه NumPy جهت رتبه‌بندی نهایی با شباهت کسینوسی بازمی‌گرداند و ابعاد جست‌وجو را از ۲۵٬۹۷۱ سند به تنها ۸۴۳ سند کاهش می‌دهد.

</div>


In [15]:
def reduce_search_space(v, planes_l, hash_tables, id_tables):
    if not (len(planes_l) == len(hash_tables) == len(id_tables)):
        raise ValueError('ابتدا سؤال سوم را کامل کنید؛ تعداد جدول‌ها باید برابر باشد.')

    candidate_vectors = []
    candidate_ids_set = set()

    for planes, hash_table, id_table in zip(planes_l, hash_tables, id_tables):
        bucket = hash_vector(v, planes)
        retrieved_vectors = hash_table[bucket]
        retrieved_ids = id_table[bucket]

        for row_id, candidate_vector in zip(retrieved_ids, retrieved_vectors):
            if row_id not in candidate_ids_set:
                candidate_vectors.append(candidate_vector)
                candidate_ids_set.add(row_id)

    print(f'Reduced space from {search_space_len} documents to {len(candidate_vectors)} documents')
    return np.asarray(candidate_vectors, dtype=complaint_embeddings.dtype).reshape(-1, embedding_dims)

<div dir=rtl>
در این مرحله جمله هدف را به embedding تبدیل کرده و الگوریتم LSH را روی آن اعمال می‌کنیم.
</div>

In [16]:
problem = 'Chest feels heavy and difficulty in breathing, I keep coughing. Sweating alot'

In [17]:
problem_embedding = model.encode(problem, convert_to_numpy=True)


In [18]:
start = time.perf_counter()
vecs_to_consider = reduce_search_space(problem_embedding, planes_l, hash_tables, id_tables)
candidate_retrieval_seconds = time.perf_counter() - start


Reduced space from 25971 documents to 843 documents


<div dir=rtl>
در ادامه با استفاده از شباهت کسینوسی، سه بردار مشابه را پیدا می‌کنیم.

### سوال پنجم) دلیل استفاده از معیار شباهت کسینوسی را توضیح دهید.
</div>

<div dir="rtl" style="text-align: right; line-height: 1.8;">

### پاسخ سؤال پنجم: دلایل انتخاب معیار شباهت کسینوسی (Cosine Similarity)

استفاده از شباهت کسینوسی در مقایسه امبدینگ‌های متنی مبتنی بر دلایل نظری و کاربردی زیر است:

1. **استقلال از طول و مقیاس متن (Scale Invariance):** مدل‌های زبانی امبدینگ (نظیر `all-mpnet-base-v2`) معنا و بار مفهومی عبارات را در جهت و زاویه بردار در فضای ۷۶۸ بعدی ذخیره می‌کنند. طول بردار (نرم اقلیدسی) اغلب تحت تأثیر طول متن، پرگویی یا تکرار کلمات قرار دارد. شباهت کسینوسی با نرمال‌سازی طول بردارها، صرفاً هم‌جهتی معنایی را مقایسه می‌نماید.
2. **سازگاری ریاضی با ابرصفحه‌های تصادفی (SimHash / Goemans-Williamson):** الگوریتم Random-Hyperplane LSH مستقیماً بر مبنای زاویه فضایی میان بردارها فرمول‌بندی شده است. احتمال اینکه یک ابرصفحه تصادفی دو بردار را از هم جدا نکند برابر است با $1 - \frac{\theta}{\pi}$ که در آن زاویه $\theta = \arccos(\text{Cosine Similarity})$ است. بنابراین معیار طبیعی، دقیق و سازگار با این روش، شباهت کسینوسی است.
3. **کارایی و سرعت محاسباتی در ابعاد بالا:** محاسبه شباهت کسینوسی بر روی بردارهای نرمال‌شده به یک ضرب داخلی ماتریسی ساده تقلیل می‌یابد که بهینگی محاسباتی بسیار بالایی روی پردازنده‌ها به همراه دارد.

</div>


In [19]:
def cosine_similarity(a, b):
    denominator = norm(a) * norm(b)
    if denominator == 0:
        return 0.0
    return float(dot(a, b) / denominator)


In [20]:
def nearest_k(v, search_embeddings, top_n=3):
    search_embeddings = np.asarray(search_embeddings).reshape(-1, len(v))
    if len(search_embeddings) == 0:
        return search_embeddings
    similarities = np.array([cosine_similarity(v, item) for item in search_embeddings])
    top_indices = np.argsort(similarities, kind='stable')[::-1][:top_n]
    return search_embeddings[top_indices]

In [21]:
start = time.perf_counter()
x = nearest_k(problem_embedding, complaint_embeddings, top_n=3)
full_search_seconds = time.perf_counter() - start
print(f'Full search: {full_search_seconds:.4f} s')


Full search: 0.0501 s


In [22]:
start = time.perf_counter()
y = nearest_k(problem_embedding, vecs_to_consider, top_n=3)
reranking_seconds = time.perf_counter() - start
lsh_search_seconds = candidate_retrieval_seconds + reranking_seconds
print(f'LSH candidate retrieval + ranking: {lsh_search_seconds:.4f} s')
print(f'Candidates: {len(vecs_to_consider)} / {len(complaint_embeddings)}')


LSH candidate retrieval + ranking: 0.0047 s
Candidates: 843 / 25971


In [23]:
def top_ids_by_cosine(query, row_ids, top_n=3):
    row_ids = np.asarray(list(row_ids), dtype=np.int64)
    if row_ids.size == 0:
        return []
    pool = complaint_embeddings[row_ids]
    q_norm = norm(query)
    norms = np.linalg.norm(pool, axis=1)
    scores = (pool @ query) / np.maximum(norms * q_norm, 1e-12)
    order = np.lexsort((row_ids, -scores))[:top_n]
    return [int(row_ids[i]) for i in order]

candidate_ids_set = set()
for planes, ids_table in zip(planes_l, id_tables):
    candidate_ids_set.update(ids_table[hash_vector(problem_embedding, planes)])

full_ids = top_ids_by_cosine(problem_embedding, range(len(complaint_embeddings)), 3)
lsh_ids = top_ids_by_cosine(problem_embedding, candidate_ids_set, 3)

def show_results(row_ids):
    for row_id in row_ids:
        print(f'ID: {row_id}')
        print(f'Review: {data.iloc[row_id]["review"]}')
        print(f'Condition: {data.iloc[row_id]["condition"]}')
        print()

In [24]:
print('=== Exact search results ===')
show_results(full_ids)

=== Exact search results ===
ID: 24942
Review: "Caused tightness in chest and nausea"
Condition: Constipation, Chronic

ID: 14260
Review: "Been feeling bad about a week after being at urgent care and breathing and chest tight."
Condition: Bacterial Infection

ID: 17166
Review: "Had a very bad cold - treated myself but then settled on my chest.  Went to Urgent Care when started to have shortness of breath and heaviness in my chest.  Given a shot of Rocephin, then put on Levaquin.  Symptoms cleared up within 48 hours."
Condition: Bronchitis



In [25]:
print('=== LSH search results ===')
show_results(lsh_ids)

=== LSH search results ===
ID: 17166
Review: "Had a very bad cold - treated myself but then settled on my chest.  Went to Urgent Care when started to have shortness of breath and heaviness in my chest.  Given a shot of Rocephin, then put on Levaquin.  Symptoms cleared up within 48 hours."
Condition: Bronchitis

ID: 9882
Review: "Diagnosed with walking pneumonia. Prescribed Avelox 400 mg once daily. After second dose, woke with chest pain and severe pressure. Discontinued medication."
Condition: Pneumonia

ID: 23338
Review: "Severe bronchitis with tight chest, wheezing, and violent coughing. Started Biaxin and noticed improvement in chest heaviness within two days."
Condition: Bronchitis




<div dir="rtl">
در بخش پایانی نتایج جست‌وجو در کل داده‌ها و فضای محدودشده با LSH را مشاهده می‌کنیم.

### سؤال ششم)
نتایج دو روش جست‌وجوی دقیق (Exact Search) و جست‌وجوی مبتنی بر LSH را از نظر اسناد پیدا‌شده، تعداد کاندیدها و زمان اجرا مقایسه و تحلیل کنید. آیا نتایج دو روش همیشه یکسان هستند؟ دلیل خود را توضیح دهید.

**نکته:** در مقایسه زمان اجرا، تنها زمان پاسخ‌گویی به پرس‌وجو را در نظر بگیرید. زمان تولید embeddingها، ساخت ابرصفحه‌ها و جدول‌های هش در این مقایسه لحاظ نشده است. زمان روش LSH شامل بازیابی کاندیدها و رتبه‌بندی آن‌هاست.
</div>


<div dir="rtl" style="text-align: right; line-height: 1.8;">

### پاسخ سؤال ششم: تحلیل و مقایسه جامع روش جست‌وجوی دقیق و LSH

#### ۱. مقایسه نتایج اسناد، تعداد کاندیدها و زمان پاسخ‌گویی:
- **روش جست‌وجوی دقیق (Exact Search):**
  - تمامی ۲۵٬۹۷۱ بردار کل دیتاست را به‌طور کامل پویش کرده و سه سند با بیشترین شباهت کسینوسی را با شناسه‌های **`[24942, 14260, 17166]`** استخراج می‌کند.
  - زمان اجرای جست‌وجو در این روش برابر **۰٫۰۵۰۱ ثانیه** است.
- **روش مبتنی بر LSH (Random-Hyperplane LSH):**
  - با اعمال ۲۵ جدول هش، فضای جست‌وجو را از ۲۵٬۹۷۱ سند به تنها **۸۴۳ سند کاندید** کاهش می‌دهد (کاهش بیش از ۹۶٫۷۵٪ فضای جست‌وجو).
  - زمان پاسخ‌گویی به **۰٫۰۰۴۶۷ ثانیه** می‌رسد که نشان‌دهنده **۱۰٫۷۲ برابر سرعت بیشتر** نسبت به روش دقیق است.
  - سند رتبه سوم روش دقیق (**سند شماره ۱۷۱۶۶**) به صورت مشترک توسط LSH بازیابی شده است. دو سند دیگر ارائه‌شده توسط LSH (شناسه‌های ۹۸۸۲ و ۲۳۳۳۸) نیز از نظر محتوای بالینی و تشخیصی با پرسمان کوئری مطابقت موضوعی بالایی دارند.

#### ۲. آیا نتایج دو روش همیشه یکسان هستند؟ چرا؟
**خیر؛ نتایج این دو روش لزوماً یکسان نیستند.** روش LSH یک الگوریتم **تقریبی (Approximate Nearest Neighbors - ANN)** و مبتنی بر تصادف است:
- در روش دقیق، همواره نزدیک‌ترین همسایه‌ها با تضمین ریاضی ۱۰۰٪ شناسایی می‌شوند زیرا ضرب داخلی تمام بردارها محاسبه می‌گردد.
- در روش LSH، برای دستیابی به سرعت بالا در کلان‌داده‌ها، مقداری خطای تفکیک تصادفی پذیرفته می‌شود. برای دو بردار با زاویه $\theta$، احتمال قرارگیری در یک باکت یکسان در جدولی با ۱۱ ابرصفحه برابر با $(1 - \frac{\theta}{\pi})^{11}$ است. با $L=25$ جدول مستقل، احتمال کاندید شدن یک سند در حداقل یکی از جدول‌ها برابر است با:

<div dir="ltr" align="center">

$$P_{\text{candidate}} = 1 - \left(1 - \left(1 - \frac{\theta}{\pi}\right)^{11}\right)^{25}$$

</div>

اگر یک همسایه واقعی در هر ۲۵ جدول توسط حداقل یک ابرصفحه تصادفی از بردار پرس‌وجو جدا شود، دچار خطای **False Negative** شده و در مرحله رتبه‌بندی نهایی حضور نخواهد داشت. با افزایش تعداد جدول‌ها ($L$) نرخ بازیابی (Recall) افزایش می‌یابد اما تعداد کاندیدها بیشتر شده و زمان پاسخ‌گویی افزایش می‌یابد؛ این موازنه بنیادین میان دقت و سرعت در پردازش داده‌های حجیم است.

</div>
